# Notebook 10: LSTMs & GRUs
**Estimated time:** 40-50 min
**Prerequisites:** Notebooks 1-9

---
## What you will learn
1. Why LSTMs were invented (vanishing gradients)
2. How the LSTM gates work (with intuition)
3. Using `nn.LSTM` in PyTorch
4. GRU — the simpler, often equally good alternative
5. Sequence classification: sentiment analysis
6. Bidirectional RNNs

## 1. LSTM — Long Short-Term Memory

**Imagine the RNN's notepad had no eraser and kept getting full of old notes.**
You can't read the new important stuff because old stuff is in the way.

The LSTM adds a **conveyor belt** (cell state) that runs through time.
Information can be **added** or **removed** from the belt using gates.

The 3 gates:
1. **Forget gate** — "What old information should I throw away?"
2. **Input gate** — "What new information should I add?"
3. **Output gate** — "What should I output right now?"

Each gate is a small neural network (sigmoid → values between 0 and 1).
0 means "block everything", 1 means "let everything through".

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Manual LSTM cell to understand each gate
class ManualLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # All 4 gates computed at once (more efficient)
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x, state):
        h, c = state   # h = hidden state, c = cell state

        combined = torch.cat([x, h], dim=1)      # concatenate input and hidden
        gates    = self.gates(combined)           # compute all gates at once

        # Split into 4 equal parts: forget, input, gate (candidate), output
        f, i, g, o = gates.chunk(4, dim=1)

        f = torch.sigmoid(f)     # forget gate: 0 = forget, 1 = keep
        i = torch.sigmoid(i)     # input gate:  0 = ignore, 1 = add
        g = torch.tanh(g)        # candidate cell values (-1 to 1)
        o = torch.sigmoid(o)     # output gate: 0 = hide, 1 = reveal

        # Update cell state: forget old + add new candidate
        c_new = f * c + i * g

        # Hidden state: tanh(cell) filtered by output gate
        h_new = o * torch.tanh(c_new)

        return h_new, c_new


# Test it
cell = ManualLSTMCell(input_size=4, hidden_size=8)
x = torch.randn(2, 4)                           # batch=2
h0 = torch.zeros(2, 8)
c0 = torch.zeros(2, 8)

for t in range(5):
    h0, c0 = cell(x, (h0, c0))
    print(f't={t}: h_norm={h0.norm():.4f}, c_norm={c0.norm():.4f}')

## 2. Using `nn.LSTM`

PyTorch's LSTM is the same as above but optimized with CUDA kernels.

The main difference from `nn.RNN`:
- Two states: `(h_n, c_n)` — hidden state AND cell state
- Initial state: pass `(h0, c0)` or leave as None for zeros

In [ ]:
lstm = nn.LSTM(
    input_size  = 16,
    hidden_size = 64,
    num_layers  = 2,
    batch_first = True,
    dropout     = 0.2,         # applied between layers (not last)
    bidirectional = False
)

x   = torch.randn(32, 20, 16)      # (batch, seq_len, features)
out, (h_n, c_n) = lstm(x)

print(f'Input  : {x.shape}')
print(f'Output : {out.shape}')     # (32, 20, 64) — all hidden states
print(f'h_n    : {h_n.shape}')     # (num_layers=2, batch=32, hidden=64)
print(f'c_n    : {c_n.shape}')     # same shape as h_n

## 3. GRU — Gated Recurrent Unit

GRU simplifies LSTM by merging the forget and input gates into one **update gate**, and removing the separate cell state.

**Result:** GRU has fewer parameters and is faster to train.
In practice, GRU often performs comparably to LSTM — always worth trying both.

In [ ]:
gru = nn.GRU(input_size=16, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)

x = torch.randn(32, 20, 16)
out, h_n = gru(x)   # only ONE state (no cell state)

print(f'GRU Output: {out.shape}')   # (32, 20, 64)
print(f'GRU h_n   : {h_n.shape}')  # (2, 32, 64)

# Parameter count comparison
lstm2 = nn.LSTM(16, 64, 2, batch_first=True)
gru2  = nn.GRU( 16, 64, 2, batch_first=True)
print(f'LSTM params: {sum(p.numel() for p in lstm2.parameters()):,}')
print(f'GRU  params: {sum(p.numel() for p in gru2.parameters()):,}')

## 4. Sentiment Analysis — Sequence Classification

**Task:** Given a short text, classify it as positive or negative.

We'll use a toy dataset (hand-crafted) to keep things simple and focus on the architecture.

In [ ]:
# Toy sentiment dataset
texts = [
    'i love this movie',      'this film is amazing',    'great acting and story',
    'i really enjoyed it',    'fantastic and wonderful',  'best movie ever seen',
    'absolutely loved it',    'brilliant and inspiring',
    'terrible movie awful',   'waste of time boring',    'hate this film',
    'really bad acting',      'worst movie ever',        'completely disappointed',
    'awful and unwatchable',  'horrible experience',
]
labels = [1]*8 + [0]*8   # 1=positive, 0=negative

# Build character vocabulary
all_chars = sorted(set(''.join(texts)))
char2id   = {c: i+1 for i, c in enumerate(all_chars)}   # 0 = padding
vocab_size = len(char2id) + 1
print(f'Vocab size: {vocab_size}')

# Encode and pad sequences
MAX_LEN = max(len(t) for t in texts)

def encode_text(text, char2id, max_len):
    ids = [char2id.get(c, 0) for c in text]
    # Pad with zeros to max_len
    ids += [0] * (max_len - len(ids))
    return ids[:max_len]

X = torch.tensor([encode_text(t, char2id, MAX_LEN) for t in texts], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)

print(f'X shape: {X.shape}')
print(f'Example encoding: {X[0]}')

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers,
                                  batch_first=True, dropout=dropout)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embedding(x)               # (batch, seq, embed_dim)
        out, (h_n, _) = self.lstm(embedded)
        last_h = self.dropout(h_n[-1])             # last layer, last step
        return self.fc(last_h).squeeze(1)          # (batch,) logits


model     = SentimentLSTM(vocab_size=vocab_size, embed_dim=16, hidden_size=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

losses = []
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    preds = model(X)
    loss  = criterion(preds, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title('Sentiment LSTM Training Loss')
plt.xlabel('Epoch')
plt.show()

In [ ]:
# Evaluate
model.eval()
with torch.no_grad():
    preds = torch.sigmoid(model(X))

print(f'{"Text":<35} {"True":>6} {"Pred":>6} {"Correct":>8}')
print('-' * 60)
for text, true, pred in zip(texts, labels, preds):
    correct = 'YES' if (pred > 0.5) == bool(true) else 'NO'
    print(f'{text[:34]:<35} {true:>6} {pred.item():>6.3f} {correct:>8}')

## 5. Bidirectional LSTMs

**Imagine reading a sentence — sometimes you need to see what comes AFTER a word to understand it.**
"I saw her **duck**" — is "duck" a bird or a verb? You need to read forward AND backward.

Bidirectional LSTM runs two LSTM networks:
- One forward (left to right)
- One backward (right to left)
Then concatenates their hidden states.

**Double the hidden size** in the output: `(batch, seq_len, 2 * hidden_size)`.

In [ ]:
bi_lstm = nn.LSTM(
    input_size=16, hidden_size=32,
    num_layers=1, batch_first=True,
    bidirectional=True
)

x = torch.randn(4, 10, 16)
out, (h_n, c_n) = bi_lstm(x)

print(f'Output shape: {out.shape}')  # (4, 10, 64) = 2 * 32 (forward + backward)
print(f'h_n    shape: {h_n.shape}')  # (2, 4, 32) — 2 directions
# To get final representation:
# h_n[0] = forward final state, h_n[1] = backward final state
h_combined = torch.cat([h_n[0], h_n[1]], dim=1)
print(f'Combined h : {h_combined.shape}')   # (4, 64)

---
## YOUR TURN — Mini Project

**Task:** Character-level sentiment classifier using GRU.

**Dataset:** Use the toy dataset from above, but this time build a word-level model.

**Steps:**
1. Build a word-level vocabulary from the texts above (split by space)
2. Encode each text as a sequence of word indices (pad to max length)
3. Build a `SentimentGRU`:
   - `nn.Embedding(vocab_size, 32)` → `nn.GRU(32, 64, num_layers=2)` → `nn.Linear(64, 1)`
4. Train for 300 epochs
5. Compare accuracy vs the character-level LSTM above
6. **Which is better — character or word level? Why?**

**Bonus challenge:** Try both LSTM and GRU at the word level and compare their loss curves.

In [ ]:
# YOUR CODE HERE
# 1. Build word vocabulary

# 2. Encode texts as word sequences

# 3. SentimentGRU class

# 4. Train for 300 epochs

# 5. Compare accuracy

# 6. Explain: char vs word level